# Llama-3-8B-Instruct Evaluation

Meta Llama-3-8B_Instruct 모델을 이용하여 다양한 태스크를 테스트해 보고, SQuAD Evaluation Dataset을 이용하여 MRC(Machine Reading Comprehension) 성능을 테스트해 보도록 하겠습니다. 마지막으로는 LLM을 이용한 번역성능 자동 평가를 실습해 보겠습니다.

## 0. Setup

In [ ]:
import os
os.environ['HF_HOME'] = 'D:/HF/cache'
os.environ['HF_DATASETS'] = 'D:/HF/datasets'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = "1"

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["FSDP_CPU_RAM_EFFICIENT_LOADING"] = "false"

In [ ]:
# !pip install -q transformers==4.51.1
# !pip install -q datasets==3.2.0

In [ ]:
from IPython.display import display
from tqdm.notebook import tqdm as notebook_tqdm

In [ ]:
"""
from huggingface_hub import notebook_login
notebook_login()
"""

In [ ]:
"""
from huggingface_hub import login
from dotenv import load_dotenv
from pathlib import Path
import os

dotenv_path = Path('Z:/Misc/access_token.env.txt')
load_dotenv(dotenv_path=dotenv_path)
access_key=os.getenv('HF_TOKEN')

print(f"HF Access Key: {access_key}")
"""

In [ ]:
!nvidia-smi

## 1. Llama-3-8B-Instruct Model

Meta Llama-3-8B-Instruct 모델을 가져오도록 하겠습니다.  
모델을 가져오기 위해서는 **Hugging Face Access Token**이 필요하며, Llama 모델 사용에 대한 사전 신청이 필요합니다.  
(https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct)

모델 접근 권한이 어려운 경우 아래의 로컬 디렉토리(Group Volume)에 있는 모델을 활용하시면 됩니다.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# model_ckpt = "meta-llama/Meta-Llama-3-8B-Instruct"
model_ckpt = "//swschoolavdazfiles002.file.core.windows.net/aias-language/Model/Meta-Llama-3-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    model_ckpt,
    torch_dtype=torch.bfloat16, # float16
    device_map="auto",
)

In [ ]:
def generate_response(system_message, user_message):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ]

    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    attention_mask = (input_ids != tokenizer.pad_token_id).long()

    terminators = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|eot_id|>")
    ]

    outputs = model.generate(
        input_ids,
        max_new_tokens=512,
        eos_token_id=terminators,
        attention_mask=attention_mask,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        temperature=0.6,
        top_p=0.9,
    )

    response = outputs[0][input_ids.shape[-1]:]

    return tokenizer.decode(response, skip_special_tokens=True)

## 2. SQuAD Evaluation

## 2.1 SQuAD Dataset

MRC(Machine Reading Comprehension) 성능평가를 위해 **SQuAD** Dataset을 가져오겠습니다.

In [ ]:
from datasets import load_dataset

dataset = load_dataset('//swschoolavdazfiles002.file.core.windows.net/aias-language/Dataset/squad')
print(dataset)

In [ ]:
dataset["train"][0]

In [ ]:
import pandas as pd

dataset.set_format(type="pandas")
df = dataset["train"][:]
df.head()

In [ ]:
dataset.reset_format()

Context, Question을 이용하여 LLM을 위한 Prompt Format으로 구성합니다.

In [ ]:
def format_context_question(data):
    return f"Context: {data['context']}\nQuestion: {data['question']}"

In [ ]:
print(format_context_question(dataset["validation"][0]))

In [ ]:
print(model.device)

In [ ]:
question_prompt = format_context_question(dataset["validation"][0])
question_prompt

In [ ]:
response = generate_response(system_message="너는 컨텍스트에 기반해서 질문에 답변을 하는 챗봇이야. 답변만 간결하게 영어로 작성해줘",
                             user_message=question_prompt)

print(response)

In [ ]:
ground_truth_text = dataset["validation"][0]["answers"]["text"]
ground_truth_text

## 2.2 SQuAD Evaluation

EM(Exact Match)-Score, F1-Score 값을 계산합니다. 특히 F1-Score는 Ground Truth 들과 일치하는 토큰 비율을 평가하게 됩니다.

In [ ]:
def normalize_text(s):
    """평가를 위하여 의미없는 Article, White Space, Punctuation 기호를 삭제합니다. """
    import string, re

    def remove_articles(text):
        regex = re.compile(r"\b(a|an|the)\b", re.UNICODE)
        return re.sub(regex, " ", text)

    def white_space_fix(text):
        return " ".join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return "".join(ch for ch in text if ch not in exclude)

    def lower(text):
        return text.lower()

    return white_space_fix(remove_articles(remove_punc(lower(s))))

def compute_exact_match(prediction, truth):
    return int(normalize_text(prediction) == normalize_text(truth))

def compute_f1(prediction, truth):
    pred_tokens = normalize_text(prediction).split()
    truth_tokens = normalize_text(truth).split()

    # Prediction 또는 Truth 값이 0 일 경우에는 두 값이 일치하면 f1 = 1 그렇지 않으면 0
    if len(pred_tokens) == 0 or len(truth_tokens) == 0:
        return int(pred_tokens == truth_tokens)

    common_tokens = set(pred_tokens) & set(truth_tokens)

    # 공통된 토큰이 없으면 f1 = 0
    if len(common_tokens) == 0:
        return 0

    prec = len(common_tokens) / len(pred_tokens)
    rec = len(common_tokens) / len(truth_tokens)

    return 2 * (prec * rec) / (prec + rec)

In [ ]:
question = dataset["validation"][0]["question"]
prediction = response
gold_answers = list(ground_truth_text)

# [실습] 다음 코드를 완성하세요!!
# 앞에서 정의한 compute_exact_match(), compute_f1() 함수를 이용하여 em_score와 f1_score를 계산합니다.
em_score = max((compute_exact_match(prediction, answer)) for answer in gold_answers)
f1_score = max((compute_f1(prediction, answer)) for answer in gold_answers)

print(f"Question: {question}")
print(f"Prediction: {prediction}")
print(f"True Answers: {gold_answers}")
print(f"EM: {em_score} \t F1: {f1_score}")

## 3. LLM Based Evaluation

LLM을 활용하여 번역 성능을 자동으로 평가하는 실습을 진행하겠습니다. (LLM as a Judge)  
System Message로 시스템의 역할을 정의하고, 정량적인 번역평가 가이드와 함께 평가 결과의 출력 형태를 제시합니다.   
User Message로는 Original Sentence, Machine Translation, Reference Translation을 각각 제시하고 가이드에 따른 평가를 요청합니다.

In [ ]:
# [실습] 다음 코드를 완성하세요!!
# LLM Based Evaluation을 위한 System Message를 작성해 보세요.
system_message = """
You are an expert in machine translation evaluation.
Your task is to assess the translation quality of a given sentence compared to a reference translation.
You will compare two translations, provide detailed feedback on the differences, and rate the translation quality on a scale from 1 to 5, where:
1 = Poor (significant errors in meaning and grammar)
2 = Fair (some errors in meaning or grammar)
3 = Good (minor issues, but overall understandable)
4 = Very good (accurate translation with minor stylistic differences)
5 = Excellent (perfect translation)

Provide your evaluation, including:
1. A brief description of the errors (if any) in terms of meaning or grammar.
2. A score from 1 to 5.
3. A suggestion to improve the translation if necessary.
"""

user_message = """
Original Sentence (English): "The quick brown fox jumps over the lazy dog."
Machine Translation (French): "Le rapide renard brun saute par-dessus le chien paresseux."
Reference Translation (French): "Le renard brun rapide saute par-dessus le chien endormi."

Evaluate the translation quality based on the given input.
"""

In [ ]:
response = generate_response(system_message=system_message, user_message=user_message)

print(response)